In [ ]:
import os

os.environ["ROBOFLOW_API_KEY"] = ""


In [ ]:
import gc
import importlib
import inspect
import os
import sys
import weakref
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import supervision as sv
import torch
import torchvision.transforms as T
from PIL import Image
from rfdetr import RFDETRBase
from rfdetr.util.misc import NestedTensor
from supervision.metrics import MeanAveragePrecision
from tqdm import tqdm

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_ROOT = Path("path/to/dataset")
OUTPUT_DIR = Path("path/to/output/cka_loss")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Training hyperparameters ──────────────────────────────────────────────────
EPOCHS = 50
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 16
LEARNING_RATE = 1e-4
CKA_LAMBDA = 0.5

# ── Inference ─────────────────────────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 0.5

print("✓ Configuration loaded")
print(f"  Dataset root : {DATASET_ROOT}")
print(f"  Output dir   : {OUTPUT_DIR}")
print(f"  Epochs: {EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LEARNING_RATE}  |  CKA λ: {CKA_LAMBDA}")

In [ ]:
CATEGORIES = [
    {"id": 0, "name": "fish", "supercategory": "animal"},
    {"id": 1, "name": "jellyfish", "supercategory": "animal"},
    {"id": 2, "name": "penguin", "supercategory": "animal"},
    {"id": 3, "name": "puffer_fish", "supercategory": "animal"},
    {"id": 4, "name": "shark", "supercategory": "animal"},
    {"id": 5, "name": "stingray", "supercategory": "animal"},
    {"id": 6, "name": "starfish", "supercategory": "animal"},
]

print(f"Dataset: Aquatic Animals  |  {len(CATEGORIES)} classes\n")
for cat in CATEGORIES:
    print(f"  [{cat['id']}] {cat['name']}")

In [ ]:
# Verify Modified `lwdetr.py` is Loaded

for mod in ["rfdetr.models.lwdetr", "rfdetr.models"]:
    if mod in sys.modules:
        del sys.modules[mod]

import rfdetr.models.lwdetr

print(f"✓ lwdetr.py loaded from: {rfdetr.models.lwdetr.__file__}")

source = inspect.getsource(rfdetr.models.lwdetr.LWDETR.forward)
if "backbone_features" in source:
    print("✓ Modified LWDETR.forward() detected — backbone_features hook is present.")
else:
    print("✗ backbone_features NOT found in LWDETR.forward().")
    print("  Ensure lwdetr.py includes the backbone feature extraction patch.")


In [ ]:
# Confirm `backbone_features` in Model Output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")

sample_images = list((DATASET_ROOT / "train" / "images").iterdir())
assert sample_images, "No images found in train/images — check DATASET_ROOT."

img = Image.open(sample_images[0]).convert("RGB").resize((448, 448))
img_tensor = T.ToTensor()(img).unsqueeze(0).to(device)
mask = torch.zeros((1, 448, 448), dtype=torch.bool, device=device)

debug_model = RFDETRBase()
debug_model.model.model = debug_model.model.model.to(device)

with torch.no_grad():
    outputs = debug_model.model.model(NestedTensor(img_tensor, mask))

print("Model output keys:", list(outputs.keys()))

if "backbone_features" in outputs:
    print("\n✓ backbone_features confirmed in model output:")
    for scale, tensor in outputs["backbone_features"].items():
        print(f"  Scale {scale}: {tensor.shape}")
else:
    print("\n✗ backbone_features missing — CKA loss will not receive feature maps.")

del debug_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        total_mem = props.total_memory / 1024 ** 3
        print(f"  GPU {i}: {props.name}  |  {total_mem:.1f} GB VRAM")

In [ ]:
model = RFDETRBase()
history = []

def _on_epoch_end(data: dict):
    history.append(data)
    map_val = data.get("test_coco_eval_bbox", [0])[0] if "test_coco_eval_bbox" in data else data.get("test_map", 0)
    message = (
        f"  Epoch {data.get('epoch', 0):>3}  |  "
        f"Train loss: {data.get('train_loss', 0):.4f}  |  "
        f"Val loss: {data.get('test_loss', 0):.4f}  |  "
        f"mAP: {map_val:.4f}"
    )
    if "train_loss_cka" in data:
        message += f"  |  CKA loss: {data['train_loss_cka']:.4f}"
    print(message)

model.callbacks["on_fit_epoch_end"].append(_on_epoch_end)

print("Model initialised — starting training with CKA loss ...\n")

model.train(
    dataset_dir=str(DATASET_ROOT),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    lr=LEARNING_RATE,
    output_dir=str(OUTPUT_DIR),
    use_ema=True,
    tensorboard=True,
    early_stopping=True,
    early_stopping_patience=10,
    amp=True,
    device="cuda",
    num_workers=0,
    multi_scale=False,
    resolution=448,
    cka_lambda=CKA_LAMBDA,
)

print(f"\n✅ Training complete  |  Checkpoints → {OUTPUT_DIR}")


In [ ]:
metrics_img_path = OUTPUT_DIR / "metrics_plot.png"

if metrics_img_path.exists():
    metrics_img = Image.open(metrics_img_path)
    plt.figure(figsize=(14, 6))
    plt.imshow(metrics_img)
    plt.axis("off")
    plt.title("Training Metrics (RF-DETR metrics_plot.png)", fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print(f"⚠ metrics_plot.png not found at {metrics_img_path}")

if history:
    df = pd.DataFrame(history)
    has_cka = "train_loss_cka" in df.columns
    ncols = 2 if has_cka else 1

    fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 4))
    if ncols == 1:
        axes = [axes]

    ax = axes[0]
    ax.plot(df["epoch"], df["train_loss"], label="Train", marker="o", linewidth=2)
    ax.plot(df["epoch"], df["test_loss"], label="Validation", marker="o", linewidth=2, linestyle="--")
    ax.set_title("Train / Validation Loss", fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

    if has_cka:
        ax = axes[1]
        ax.plot(df["epoch"], df["train_loss_cka"], color="#e74c3c", label="CKA Loss", marker="d", linewidth=2)
        ax.set_title("CKA Contrastive Loss", fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("CKA Loss")
        ax.legend()
        ax.grid(True, alpha=0.3)

    fig.suptitle(f"RF-DETR + CKA Loss (λ={CKA_LAMBDA}) — Aquatic Dataset", fontsize=13, fontweight="bold")
    plt.tight_layout()
    save_path = OUTPUT_DIR / "training_curves.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved → {save_path}")

In [ ]:
def cleanup_gpu(obj=None, verbose: bool = True) -> None:
    if not torch.cuda.is_available():
        print("CUDA not available — nothing to clean up.")
        return

    def _stats():
        return (
            torch.cuda.memory_allocated() / 1024 ** 2,
            torch.cuda.memory_reserved() / 1024 ** 2,
        )

    torch.cuda.synchronize()
    if verbose:
        alloc, reserv = _stats()
        print(f"Before  →  Allocated: {alloc:.1f} MB  |  Reserved: {reserv:.1f} MB")

    if obj is not None:
        ref = weakref.ref(obj)
        del obj
        if ref() is not None and verbose:
            print("⚠ Object still referenced elsewhere — partial cleanup only.")

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = _stats()
        print(f"After   →  Allocated: {alloc:.1f} MB  |  Reserved: {reserv:.1f} MB")

cleanup_gpu(model, verbose=True)


In [ ]:
CHECKPOINT = OUTPUT_DIR / "checkpoint_best_total.pth"

model = RFDETRBase(pretrain_weights=str(CHECKPOINT))
model.optimize_for_inference()
print(f"✓ Model loaded from: {CHECKPOINT}")

ds = sv.DetectionDataset.from_coco(
    images_directory_path=str(DATASET_ROOT / "test"),
    annotations_path=str(DATASET_ROOT / "test" / "_annotations.coco.json"),
)
print(f"✓ Test set: {len(ds)} images  |  Classes: {ds.classes}")

In [ ]:
targets, predictions = [], []

for path, _, annotations in tqdm(ds, desc="Evaluating"):
    image = Image.open(path).convert("RGB")
    detections = model.predict(image, threshold=0)
    targets.append(annotations)
    predictions.append(detections)

metric = MeanAveragePrecision()
metric.update(predictions, targets)
map_result = metric.compute()

print("\n" + "=" * 45)
print(f"TEST RESULTS — Aquatic + CKA Loss (λ={CKA_LAMBDA})")
print("=" * 45)
print(f"  mAP@0.5:0.95 : {map_result.map50_95:.4f}")
print(f"  mAP@0.5      : {map_result.map50:.4f}")
print(f"  mAP@0.75     : {map_result.map75:.4f}")


In [ ]:
path, _, annotations = ds[0]
image = Image.open(path).convert("RGB")
detections = model.predict(image, threshold=CONFIDENCE_THRESHOLD)

print(f"Image : {Path(path).name}")
print(f"GT    : {len(annotations)} objects")
print(f"Pred  : {len(detections)} detections  (threshold={CONFIDENCE_THRESHOLD})")

text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
thickness = sv.calculate_optimal_line_thickness(resolution_wh=image.size)
palette = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff66ff", "#3399ff",
    "#ff66b2", "#ff8080", "#b266ff",
])

bbox_ann = sv.BoxAnnotator(color=palette, thickness=thickness)
label_ann = sv.LabelAnnotator(
    color=palette,
    text_color=sv.Color.BLACK,
    text_scale=text_scale,
)

gt_labels = [ds.classes[c] for c in annotations.class_id]
pred_labels = [f"{ds.classes[c]} {conf:.2f}" for c, conf in zip(detections.class_id, detections.confidence)]

gt_img = label_ann.annotate(bbox_ann.annotate(image.copy(), annotations), annotations, gt_labels)
pred_img = label_ann.annotate(bbox_ann.annotate(image.copy(), detections), detections, pred_labels)

sv.plot_images_grid(
    images=[gt_img, pred_img],
    grid_size=(1, 2),
    titles=["Ground Truth", f"RF-DETR + CKA (conf ≥ {CONFIDENCE_THRESHOLD})"],
)